# Prediction Horizon Curve Comparison

This notebook plots one matched forecast window for each dataset and prediction length. The x-axis is prediction horizon `1..pred_len`, not the aggregated test-set time index. Baseline windows are matched to the DARNet reference true curve when raw window order differs.


In [1]:

from pathlib import Path
from collections import defaultdict, OrderedDict
import csv
import math
import re

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

ROOT = Path.cwd().parent if Path.cwd().name == 'PredictionPlot' else Path.cwd()
DRAW_DIR = ROOT / 'draw'
OUT_DIR = ROOT / 'PredictionPlot'
FIG_DIR = OUT_DIR / 'figures_new'
DATA_DIR = OUT_DIR / 'plot_data_new'
OUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

DATASETS = ['Abilene', 'Geant', 'Seattle']
PRED_LENS = [5, 10, 15, 20]
D_MODEL = 256
TARGET_COL = 'TC0'
MODEL_DIR_TO_LABEL = OrderedDict([('net', 'DARNet'), ('PMDformer', 'PMDformer'), ('iTransformer', 'iTransformer'), ('FEDformer', 'FEDformer'), ('FeTS', 'FeTS'), ('HMformer', 'HMformer'), ('PatchTST', 'PatchTST'), ('timesnet', 'TimesNet'), ('WPMixer', 'WPMixer'), ('P_sLSTM', 'P_sLSTM'), ('xLSTMTime', 'xLSTMTime'), ('xlstm_mixer', 'xLSTM-Mixer')])
MODEL_ORDER = list(MODEL_DIR_TO_LABEL.values())
DISPLAY_ORDER = ['True'] + MODEL_ORDER
LINE_STYLE = {'True': dict(color='black', linewidth=2.8, linestyle='-', marker='o', markersize=5.0, alpha=0.95, zorder=20), 'DARNet': dict(color='#d62728', linewidth=2.4, linestyle='-', marker='s', markersize=4.8, alpha=0.96, zorder=18), 'PMDformer': dict(color='#1f77b4', linewidth=1.35, linestyle='-', marker='.', markersize=4.5, alpha=0.88), 'iTransformer': dict(color='#ff7f0e', linewidth=1.35, linestyle='-', marker='.', markersize=4.5, alpha=0.88), 'FEDformer': dict(color='#2ca02c', linewidth=1.35, linestyle='-', marker='.', markersize=4.5, alpha=0.88), 'FeTS': dict(color='#9467bd', linewidth=1.35, linestyle='-', marker='.', markersize=4.5, alpha=0.88), 'HMformer': dict(color='#8c564b', linewidth=1.35, linestyle='-', marker='.', markersize=4.5, alpha=0.88), 'PatchTST': dict(color='#e377c2', linewidth=1.35, linestyle='-', marker='.', markersize=4.5, alpha=0.88), 'TimesNet': dict(color='#7f7f7f', linewidth=1.35, linestyle='-', marker='.', markersize=4.5, alpha=0.88), 'WPMixer': dict(color='#bcbd22', linewidth=1.35, linestyle='-', marker='.', markersize=4.5, alpha=0.88), 'P_sLSTM': dict(color='#17becf', linewidth=1.35, linestyle='-', marker='.', markersize=4.5, alpha=0.88), 'xLSTMTime': dict(color='#00429d', linewidth=1.35, linestyle='--', marker='.', markersize=4.5, alpha=0.9), 'xLSTM-Mixer': dict(color='#93003a', linewidth=1.35, linestyle='--', marker='.', markersize=4.5, alpha=0.9)}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.unicode_minus': False, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'figure.dpi': 180})

def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_')

def discover_files():
    records = []
    for p in DRAW_DIR.rglob('test_raw.csv'):
        try:
            dataset, data_tag, model_dir, pl_dm, tc, filename = p.relative_to(DRAW_DIR).parts
        except ValueError:
            continue
        if dataset not in DATASETS or model_dir not in MODEL_DIR_TO_LABEL or tc != TARGET_COL:
            continue
        m = re.fullmatch(r'PL(\d+)_DM(\d+)', pl_dm)
        if not m:
            continue
        pred_len, d_model = int(m.group(1)), int(m.group(2))
        if pred_len not in PRED_LENS or d_model != D_MODEL:
            continue
        records.append({'dataset': dataset, 'data_tag': data_tag, 'model_dir': model_dir, 'model': MODEL_DIR_TO_LABEL[model_dir], 'pred_len': pred_len, 'd_model': d_model, 'target_col': tc, 'path': p})
    return records

def read_raw_windows(path, pred_len):
    df = pd.read_csv(path)
    true = pd.to_numeric(df['true'], errors='coerce').to_numpy(dtype=np.float64)
    pred = pd.to_numeric(df['pred'], errors='coerce').to_numpy(dtype=np.float64)
    usable = (len(true) // pred_len) * pred_len
    return true[:usable].reshape(-1, pred_len), pred[:usable].reshape(-1, pred_len)

def choose_reference_window(true_windows):
    safe = np.where(np.isfinite(true_windows), true_windows, np.nan)
    ranges = np.nanmax(safe, axis=1) - np.nanmin(safe, axis=1)
    peaks = np.nanmax(np.abs(safe), axis=1)
    score = np.nan_to_num(ranges, nan=-np.inf) + 0.10 * np.nan_to_num(peaks, nan=0.0)
    return int(np.nanargmax(score)) if np.any(np.isfinite(score)) else 0

def match_window_by_true(true_windows, reference_true):
    valid = np.isfinite(true_windows) & np.isfinite(reference_true[None, :])
    diff = np.where(valid, true_windows - reference_true[None, :], 0.0)
    valid_count = valid.sum(axis=1)
    mse = np.full(true_windows.shape[0], np.inf, dtype=np.float64)
    ok = valid_count > 0
    mse[ok] = (diff[ok] ** 2).sum(axis=1) / valid_count[ok]
    idx = int(np.argmin(mse))
    rmse = float(np.sqrt(mse[idx])) if np.isfinite(mse[idx]) else math.inf
    max_abs = float(np.nanmax(np.abs(true_windows[idx] - reference_true)))
    return idx, rmse, max_abs

def build_horizon_curve(records_for_combo, pred_len):
    by_model = {r['model']: r for r in records_for_combo}
    true_windows_by_model = {}
    pred_windows_by_model = {}
    for model in MODEL_ORDER:
        true_w, pred_w = read_raw_windows(by_model[model]['path'], pred_len)
        true_windows_by_model[model] = true_w
        pred_windows_by_model[model] = pred_w
    ref_window = choose_reference_window(true_windows_by_model['DARNet'])
    reference_true = true_windows_by_model['DARNet'][ref_window]
    merged = pd.DataFrame({'horizon': np.arange(1, pred_len + 1), 'True': reference_true})
    match_rows = []
    for model in MODEL_ORDER:
        if model == 'DARNet':
            matched_window, rmse, max_abs = ref_window, 0.0, 0.0
        else:
            matched_window, rmse, max_abs = match_window_by_true(true_windows_by_model[model], reference_true)
        merged[model] = pred_windows_by_model[model][matched_window]
        match_rows.append({'model': model, 'matched_window': matched_window, 'true_match_rmse': rmse, 'true_match_max_abs': max_abs, 'window_count': true_windows_by_model[model].shape[0]})
    return merged, ref_window, match_rows

def plot_horizon_curve(merged, dataset, pred_len, ref_window, out_pdf, out_png):
    fig, ax = plt.subplots(figsize=(10.6, 5.2))
    x = merged['horizon'].to_numpy()
    for name in DISPLAY_ORDER:
        if name in merged.columns:
            ax.plot(x, merged[name].to_numpy(), label=name, **LINE_STYLE.get(name, {}))
    ax.set_title(f'{dataset} Forecast Horizon Comparison (PredLen={pred_len}, RefWindow={ref_window})', fontsize=13, pad=10)
    ax.set_xlabel('Prediction Horizon')
    ax.set_ylabel('Value')
    ax.set_xticks(np.arange(1, pred_len + 1))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.38)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(ncol=7, loc='upper center', bbox_to_anchor=(0.5, -0.14), frameon=False, fontsize=9, handlelength=2.2, columnspacing=1.0)
    fig.tight_layout(rect=[0, 0.08, 1, 1])
    fig.savefig(out_pdf, bbox_inches='tight')
    fig.savefig(out_png, bbox_inches='tight', dpi=300)
    plt.close(fig)


In [2]:

records = discover_files()
print(f'Discovered eligible raw files: {len(records)}')
by_combo = defaultdict(list)
for r in records:
    by_combo[(r['dataset'], r['pred_len'])].append(r)

index_rows, quality_rows, match_rows_all = [], [], []
for dataset in DATASETS:
    for pred_len in PRED_LENS:
        items = by_combo.get((dataset, pred_len), [])
        present = sorted({r['model'] for r in items})
        missing = [m for m in MODEL_ORDER if m not in present]
        if missing:
            quality_rows.append({'dataset': dataset, 'pred_len': pred_len, 'status': 'missing', 'reference_window': '', 'horizon_count': pred_len, 'max_true_match_rmse': '', 'max_true_match_abs': '', 'missing_models': ';'.join(missing)})
            continue
        merged, ref_window, match_rows = build_horizon_curve(items, pred_len)
        for row in match_rows:
            match_rows_all.append({'dataset': dataset, 'pred_len': pred_len, 'reference_window': ref_window, **row})
        max_rmse = max(r['true_match_rmse'] for r in match_rows)
        max_abs = max(r['true_match_max_abs'] for r in match_rows)
        data_path = DATA_DIR / f'{safe_name(dataset)}_PL{pred_len}_refwindow{ref_window}_horizon_prediction_curves.csv'
        merged.to_csv(data_path, index=False, encoding='utf-8-sig')
        pdf_path = FIG_DIR / f'{safe_name(dataset)}_PL{pred_len}_horizon_prediction_curves.pdf'
        png_path = FIG_DIR / f'{safe_name(dataset)}_PL{pred_len}_horizon_prediction_curves.png'
        plot_horizon_curve(merged, dataset, pred_len, ref_window, pdf_path, png_path)
        index_rows.append({'dataset': dataset, 'pred_len': pred_len, 'reference_window': ref_window, 'pdf': str(pdf_path.relative_to(OUT_DIR)).replace('\\', '/'), 'png': str(png_path.relative_to(OUT_DIR)).replace('\\', '/'), 'merged_csv': str(data_path.relative_to(OUT_DIR)).replace('\\', '/'), 'horizon_count': len(merged)})
        quality_rows.append({'dataset': dataset, 'pred_len': pred_len, 'status': 'ok', 'reference_window': ref_window, 'horizon_count': len(merged), 'max_true_match_rmse': f'{max_rmse:.12g}', 'max_true_match_abs': f'{max_abs:.12g}', 'missing_models': ''})

with (OUT_DIR / 'prediction_horizon_figure_index.csv').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['dataset', 'pred_len', 'reference_window', 'pdf', 'png', 'merged_csv', 'horizon_count'])
    writer.writeheader(); writer.writerows(index_rows)
with (OUT_DIR / 'prediction_horizon_quality_check.csv').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['dataset', 'pred_len', 'status', 'reference_window', 'horizon_count', 'max_true_match_rmse', 'max_true_match_abs', 'missing_models'])
    writer.writeheader(); writer.writerows(quality_rows)
with (OUT_DIR / 'prediction_horizon_window_matches.csv').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['dataset', 'pred_len', 'reference_window', 'model', 'matched_window', 'true_match_rmse', 'true_match_max_abs', 'window_count'])
    writer.writeheader(); writer.writerows(match_rows_all)
print(f'Generated horizon figures: {len(index_rows)} PDF + {len(index_rows)} PNG')
print(pd.DataFrame(quality_rows))


Discovered eligible raw files: 144
Generated horizon figures: 12 PDF + 12 PNG
    dataset  pred_len status  reference_window  horizon_count  \
0   Abilene         5     ok               377              5   
1   Abilene        10     ok               372             10   
2   Abilene        15     ok               367             15   
3   Abilene        20     ok               367             20   
4     Geant         5     ok               353              5   
5     Geant        10     ok               344             10   
6     Geant        15     ok               352             15   
7     Geant        20     ok               352             20   
8   Seattle         5     ok                 5              5   
9   Seattle        10     ok                 0             10   
10  Seattle        15     ok                 0             15   
11  Seattle        20     ok                 6             20   

   max_true_match_rmse max_true_match_abs missing_models  
0    0.0002295418